# Data Preparation & Validation

This notebook loads and validates all raw datasets, builds master dataframes (human, model, BIO), and exports clean data for downstream analysis.

**Run this once at the start** before running other analysis notebooks.

**Outputs:**
- Validated human_master, model_master, bio_master dataframes
- Summary statistics and quality checks

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

import pandas as pd
import numpy as np
from src.data_loaders import load_human_master, load_model_master, load_bio_master
from src.config import get_paths

paths = get_paths()
print(f"✓ Data directory: {paths.data_dir}")
print(f"✓ Outputs directory: {paths.outputs_dir}")

## 1. Load Human Data

In [ ]:
human_master = load_human_master()
print(f"✓ Loaded human data: {human_master.shape[0]} trials, {human_master['participantID'].nunique()} participants")
print(f"  Conditions: {sorted(human_master['condition'].unique())}")
print(f"  Columns: {list(human_master.columns)}")
human_master.head()

### Human Data Summary Statistics

In [ ]:
print("Trials per participant:")
print(human_master.groupby('participantID').size().sort_values(ascending=False))
print("\nTrial outcome distribution (TP):")
print(human_master['TP'].value_counts().sort_index())
print("\nDecision distribution:")
print(human_master['decision'].value_counts().sort_index())

## 2. Load Model Data

In [ ]:
model_master = load_model_master()
print(f"✓ Loaded model data: {model_master.shape[0]} trials, {model_master['participantID'].nunique()} models")
print(f"  Models: {sorted(model_master['participantID'].unique())}")
print(f"  Columns: {list(model_master.columns)}")
model_master.head()

### Model Data Summary Statistics

In [ ]:
print("Trials per model:")
print(model_master.groupby('participantID').size().sort_values(ascending=False))
print("\nDecision distribution across models:")
print(model_master.groupby('participantID')['decision'].value_counts().sort_index())

## 3. Load BIO (Bayesian Ideal Observer) Data

In [ ]:
bio_master = load_bio_master()
print(f"✓ Loaded BIO data: {bio_master.shape[0]} trials")
print(f"  Columns: {list(bio_master.columns)}")
bio_master.head()

### BIO Performance Summary

In [ ]:
print(f"BIO decision distribution:")
print(bio_master['bio_decision'].value_counts().sort_index())
print(f"\nBIO accuracy: {bio_master['bio_outcome'].mean():.3f}")
print(f"BIO p_present (mean): {bio_master['bio_p_present'].mean():.3f}")

## 4. Cross-Domain Consistency Checks

In [ ]:
# Verify all domains have same stimIDs and conditions
human_stims = set(human_master['stimID'].unique())
model_stims = set(model_master['stimID'].unique())
bio_stims = set(bio_master['stimID'].unique())

print(f"✓ Human stimIDs: {len(human_stims)}")
print(f"✓ Model stimIDs: {len(model_stims)}")
print(f"✓ BIO stimIDs: {len(bio_stims)}")
print(f"✓ All match: {human_stims == model_stims == bio_stims}")

# Check condition alignment
print(f"\n✓ Conditions:")
print(f"  Human: {sorted(human_master['condition'].unique())}")
print(f"  Model: {sorted(model_master['condition'].unique())}")
print(f"  BIO: {sorted(bio_master['condition'].unique())}")

## 5. Export Clean Data (Optional)
Save processed dataframes for reference or archival.

In [ ]:
# Uncomment to export clean data
# human_master.to_csv(paths.outputs_dir / "human_master.csv", index=False)
# model_master.to_csv(paths.outputs_dir / "model_master.csv", index=False)
# bio_master.to_csv(paths.outputs_dir / "bio_master.csv", index=False)
# print("✓ Exported clean dataframes to outputs/")

print("✓ Data preparation complete. Ready for analysis.")